In [1]:
import os

In [2]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\Data analysis\\Kaggle_Competition\\Notebook'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\Data analysis\\Kaggle_Competition'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataExtractionConfig:
    root_dir: Path
    file_id: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from etl.constants import *
from etl.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH):
        
        self.config = read_yaml(config_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_extraction_config(self) -> DataExtractionConfig:
        config = self.config.data_extraction

        create_directories([config.root_dir])

        data_extraction_config = DataExtractionConfig(
            root_dir=config.root_dir,
            file_id=config.file_id,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_extraction_config

In [7]:
import os
import gdown
from pathlib import Path
from urllib.error import URLError
from time import sleep
from etl import logger
from etl.entity.config_entity import DataExtractionConfig  
import zipfile

In [8]:
class DataExtraction:

    def __init__(self, config):
        self.config = config

    def download_file(self, retries: int = 3, delay: int = 5):
        try:
            file_path = Path(self.config.local_data_file)

            # Ensure directory exists
            file_path.parent.mkdir(parents=True, exist_ok=True)

            if not self.config.file_id:
                raise ValueError("file_id is missing in config")

            # Skip if already exists
            if file_path.exists() and file_path.stat().st_size > 0:
                logger.info(f"File already exists: {file_path}")
                return

            url = f"https://drive.google.com/uc?id={self.config.file_id}"

            for attempt in range(1, retries + 1):
                try:
                    logger.info(f"Downloading (Attempt {attempt}) from {url}")

                    gdown.download(url, str(file_path), quiet=False, fuzzy=True)

                    if not file_path.exists() or file_path.stat().st_size == 0:
                        raise Exception("Downloaded file is empty")

                    logger.info(f"Download successful: {file_path}")
                    break

                except Exception as e:
                    logger.error(f"Attempt {attempt} failed: {e}")

                    if attempt < retries:
                        sleep(delay)
                    else:
                        raise RuntimeError("Download failed after retries")

        except Exception as e:
            logger.exception(f"Download error: {e}")
            raise

    def extract_zip_file(self):
        try:
            zip_path = self.config.local_data_file
            unzip_path = self.config.unzip_dir

            os.makedirs(unzip_path, exist_ok=True)

            # Validate zip file
            if not zipfile.is_zipfile(zip_path):
                raise ValueError("Invalid ZIP file")

            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(unzip_path)

            logger.info(f"Extraction completed at {unzip_path}")

        except Exception as e:
            logger.exception(f"Extraction error: {e}")
            raise

In [9]:
try:
    config = ConfigurationManager()
    data_extraction_config = config.get_data_extraction_config()
    data_extraction = DataExtraction(config=data_extraction_config)
    data_extraction.download_file()
    data_extraction.extract_zip_file()
except Exception as e:
    raise e

[2026-03-31 23:33:39,933: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-31 23:33:39,937: INFO: common: created directory at: artifacts]
[2026-03-31 23:33:39,940: INFO: common: created directory at: artifacts/data_extraction]
[2026-03-31 23:33:39,943: INFO: 4128518551: Downloading (Attempt 1) from https://drive.google.com/uc?id=1YrM3IEYjeYIu-Bx1lslPfDc45GWkHWEY]


Downloading...
From: https://drive.google.com/uc?id=1YrM3IEYjeYIu-Bx1lslPfDc45GWkHWEY
To: c:\Users\HP\Desktop\Python\Vs_Python\Data analysis\Kaggle_Competition\artifacts\data_extraction\data.zip
100%|██████████| 933k/933k [00:02<00:00, 352kB/s]

[2026-03-31 23:33:46,528: INFO: 4128518551: Download successful: artifacts\data_extraction\data.zip]
[2026-03-31 23:33:46,664: INFO: 4128518551: Extraction completed at artifacts/data_extraction]
